# 04 — Combined: LeRobot + Talkbot Coaching Session

**Purpose**: Run LeRobot and Talkbot together for a full voice-coached demonstration session. The student talks to talkbot while physically coaching the arm — narrate what you're doing, ask for counts, get encouragement.

**Prereqs**:
- `01_reserve_node` complete (training node provisioned)
- `02_lerobot` familiar (LeRobot image on Pi, arms calibrated)
- `03_talkbot` complete (talkbot installed with coaching prompt)
- `.env` fully populated

**Outcome**: Dataset pushed to HuggingFace Hub; talkbot session log saved; benchmark timing recorded. Hand off to `02_lerobot` to train on the collected data.

---

In [ ]:
import os
import re
import shlex
import time
import json
import subprocess
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from tqdm.notebook import tqdm

load_dotenv(dotenv_path=Path('..') / '.env', override=False)

# Benchmark: record start time
_bench = {
    "notebook": "04_lerobot_talkbot",
    "started_at": datetime.utcnow().isoformat(),
    "timings": {},
    "config": {}
}
_t0 = time.monotonic()

PI_HOST      = os.getenv("PI_HOST", "192.168.4.191")
PI_PORT      = os.getenv("PI_PORT", "22222")
PI_USER      = "root"
FLOATING_IP  = os.getenv("CONTROL_FLOATING_IP")
HF_USER      = os.getenv("HF_USER")
HF_TOKEN     = os.getenv("HF_TOKEN")
DATASET_REPO = os.getenv("DATASET_REPO_ID", f"{HF_USER}/soarm101-pick-block")
POLICY       = os.getenv("POLICY", "act")
LLM_BACKEND  = os.getenv("TALKBOT_LLM_BACKEND", "local")
PI_IMAGE     = f"{HF_USER}/lerobot-soarm101:latest"

# Session-specific dataset tag (keeps combined-session datasets separate)
SESSION_DATE = datetime.utcnow().strftime("%Y%m%d")
SESSION_REPO = f"{DATASET_REPO}-{SESSION_DATE}"

_bench["config"] = {
    "pi": PI_HOST, "dataset": SESSION_REPO,
    "llm_backend": LLM_BACKEND, "node_ip": FLOATING_IP,
}

def pi_run(cmd, capture=False, input_data=None, stream=False):
    """Run a command on the Pi over SSH."""
    ssh_prefix = [
        "ssh", "-p", PI_PORT,
        "-o", "StrictHostKeyChecking=no",
        "-o", "ConnectTimeout=10",
        f"{PI_USER}@{PI_HOST}",
    ]
    full = ssh_prefix + ["bash", "-c", cmd]
    if stream:
        proc = subprocess.Popen(full, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            print(line, end="")
        return proc.wait()
    elif capture:
        return subprocess.check_output(full, text=True).strip()
    else:
        return subprocess.run(full, input=input_data, capture_output=True, text=True)

print(f"Pi:      {PI_HOST}:{PI_PORT}")
print(f"Node:    {FLOATING_IP}")
print(f"Dataset: {SESSION_REPO}")
print(f"LLM:     {LLM_BACKEND}")

## 1. Pre-flight: Verify Both Services Ready

In [ ]:
_t_preflight = time.monotonic()
checks = {}

# Pi SSH
r = subprocess.run(
    ["ssh", "-p", PI_PORT, "-o", "ConnectTimeout=5",
     "-o", "StrictHostKeyChecking=no", f"{PI_USER}@{PI_HOST}", "echo ok"],
    capture_output=True, text=True
)
checks["pi_ssh"] = r.returncode == 0

# Serial ports
serials = pi_run("ls /dev/ttyACM* 2>/dev/null | wc -l", capture=True)
checks["serial_ports"] = int(serials or 0) >= 2

# Cameras
cams = pi_run("ls /dev/video0 /dev/video2 2>/dev/null | wc -l", capture=True)
checks["cameras"] = int(cams or 0) >= 2

# Talkbot installed
tb = pi_run("test -d ~/talkbot && echo ok || echo missing", capture=True)
checks["talkbot_installed"] = tb == "ok"

# Training node (optional)
if FLOATING_IP and FLOATING_IP not in ("REPLACE_ME_AFTER_PROVISION", "", None):
    r2 = subprocess.run(
        ["ssh", "-o", "ConnectTimeout=5", "-o", "StrictHostKeyChecking=no",
         f"cc@{FLOATING_IP}", "echo ok"],
        capture_output=True, text=True
    )
    checks["training_node"] = r2.returncode == 0
else:
    checks["training_node"] = "skipped"

all_ok = all(v is True or v == "skipped" for v in checks.values())
for name, status in checks.items():
    icon = "OK" if status is True else ("--" if status == "skipped" else "FAIL")
    print(f"  [{icon}] {name}")

if not all_ok:
    print("\nFix failing checks before starting the session.")
    print("  serial_ports/cameras: check arms and cameras are plugged into Pi")
    print("  talkbot_installed:    run 03_talkbot.ipynb first")

_bench["preflight"] = checks
_bench["timings"]["preflight_s"] = round(time.monotonic() - _t_preflight, 1)

## 2. Start Talkbot

In [ ]:
_t_talkbot = time.monotonic()

AGENT_PROMPT = os.getenv(
    "TALKBOT_AGENT_PROMPT",
    "You are a coaching assistant for a robot arm. "
    "Help the student record quality demonstration episodes."
)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")

env_parts = [
    f"TALKBOT_LLM_BACKEND={LLM_BACKEND}",
    "TALKBOT_AGENT_PROMPT=$(cat ~/talkbot/prompts/robot_coach.txt)",
]
if LLM_BACKEND == "openrouter" and OPENROUTER_API_KEY:
    env_parts.append(f"OPENROUTER_API_KEY={OPENROUTER_API_KEY}")

env_str = " ".join(env_parts)
tb_cmd = f"cd ~/talkbot && export PATH=$HOME/.local/bin:$PATH && {env_str} uv run talkbot"

result = pi_run(
    f"tmux kill-session -t talkbot 2>/dev/null || true; "
    f"tmux new-session -d -s talkbot '{tb_cmd}' && echo launched",
    capture=True
)
print(f"Talkbot: {'started' if 'launched' in result else result}")

time.sleep(2)
tb_out = pi_run("tmux capture-pane -pt talkbot -S -5 2>/dev/null", capture=True)
if tb_out:
    print(f"Output: {tb_out.splitlines()[-1] if tb_out.strip() else '(starting...)'}")

_bench["timings"]["talkbot_start_s"] = round(time.monotonic() - _t_talkbot, 1)

## 3. Start LeRobot Recording Session

In [ ]:
_t_lerobot = time.monotonic()

ROBOT        = "alpha"
DATASET_SLUG = SESSION_REPO.split("/")[-1]   # last part of repo ID
TASK_DESC    = "Pick up the block and place it in the bowl"
NUM_EPISODES = 50
EPISODE_TIME = 30
RESET_TIME   = 10

lr_cmd = (
    f"balena run -it --privileged "
    f"--device=/dev/ttyACM0 --device=/dev/ttyACM1 "
    f"--device=/dev/video0 --device=/dev/video2 "
    f"-v /tmp/fleet.yaml:/app/config/fleet.yaml "
    f"-v /mnt/data/calibration:/app/calibration "
    f"-v /mnt/data/datasets:/app/data "
    f"{PI_IMAGE} "
    f"coachable --fleet /app/config/fleet.yaml collect "
    f"--robot {ROBOT} --dataset {DATASET_SLUG} "
    f"--episodes {NUM_EPISODES} --episode-time {EPISODE_TIME} --reset-time {RESET_TIME} "
    f"--task '{TASK_DESC}' --no-push"
)

# Run in a second tmux session so talkbot and lerobot run side-by-side
result = pi_run(
    f"tmux kill-session -t lerobot 2>/dev/null || true; "
    f"tmux new-session -d -s lerobot '{lr_cmd}' && echo launched",
    capture=True
)
print(f"LeRobot: {'started' if 'launched' in result else result}")

total_s = NUM_EPISODES * (EPISODE_TIME + RESET_TIME) - RESET_TIME
print(f"Session: {NUM_EPISODES} episodes × {EPISODE_TIME}s = ~{total_s//60}m {total_s%60}s")
print(f"Dataset: {SESSION_REPO}")
print()
print(f"Monitor:  ssh -p {PI_PORT} {PI_USER}@{PI_HOST}")
print(f"          tmux attach -t lerobot   # see episode progress")
print(f"          tmux attach -t talkbot   # speak to coaching assistant")

_bench["timings"]["lerobot_start_s"] = round(time.monotonic() - _t_lerobot, 1)
_bench["config"]["num_episodes"] = NUM_EPISODES

## 4. Monitor Session Progress

Polls LeRobot episode count every 30 seconds. talkbot runs independently — talk to it anytime.

In [ ]:
POLL_INTERVAL = 30   # seconds between checks
MAX_POLLS     = 120  # poll for up to 1h then stop (re-run cell to continue)

episodes_recorded = 0

with tqdm(total=NUM_EPISODES, unit="ep", desc="Episodes") as pbar:
    for poll in range(MAX_POLLS):
        # Grab recent lerobot output and parse episode count
        lr_out = pi_run(
            "tmux capture-pane -pt lerobot -S -30 2>/dev/null || echo 'session ended'",
            capture=True
        )

        if "session ended" in lr_out or "no server running" in lr_out.lower():
            print("\nLeRobot session ended.")
            break

        # Parse lines like "Episode 12/50" or "Recorded episode 12"
        ep_matches = re.findall(r'[Ee]pisode[^\d]*(\d+)', lr_out)
        if ep_matches:
            current = int(ep_matches[-1])
            delta = current - episodes_recorded
            if delta > 0:
                pbar.update(delta)
                episodes_recorded = current

        # Show latest talkbot exchange
        tb_out = pi_run(
            "tmux capture-pane -pt talkbot -S -3 2>/dev/null",
            capture=True
        )
        last_tb = tb_out.strip().splitlines()[-1] if tb_out.strip() else ""
        pbar.set_postfix_str(f"coach: {last_tb[:50]}" if last_tb else "")

        if episodes_recorded >= NUM_EPISODES:
            print(f"\nAll {NUM_EPISODES} episodes complete!")
            break

        time.sleep(POLL_INTERVAL)

_bench["episodes_recorded"] = episodes_recorded

## 5. End Session: Stop Services and Push Dataset

In [ ]:
_t_end = time.monotonic()

# Stop LeRobot gracefully (Ctrl-C triggers dataset push if --no-push was omitted)
pi_run("tmux send-keys -t lerobot C-c", capture=True)
time.sleep(5)

# Verify lerobot stopped
lr_sessions = pi_run("tmux list-sessions 2>/dev/null | grep lerobot || echo none", capture=True)
print(f"LeRobot session: {lr_sessions}")

# Stop talkbot
pi_run("tmux send-keys -t talkbot C-c", capture=True)
time.sleep(2)

# Push dataset to HuggingFace Hub
print(f"\nPushing dataset {SESSION_REPO} to HuggingFace Hub...")
push_rc = pi_run(
    f"balena run --rm "
    f"-v /mnt/data/datasets:/app/data "
    f"-e HF_TOKEN={HF_TOKEN} "
    f"{PI_IMAGE} "
    f"huggingface-cli upload {SESSION_REPO} "
    f"/app/data/{SESSION_REPO.replace('/', os.sep)} "
    f"--repo-type dataset",
    stream=True
)
if push_rc == 0:
    print(f"\nDataset live: https://huggingface.co/datasets/{SESSION_REPO}")
else:
    print(f"\nPush exited {push_rc} — check output above.")

print()
print("Next: open 02_lerobot.ipynb and set DATASET_REPO_ID to:")
print(f"  {SESSION_REPO}")

_bench["timings"]["end_s"] = round(time.monotonic() - _t_end, 1)

## Benchmark: Save Timing

In [ ]:
_bench["timings"]["total_s"] = round(time.monotonic() - _t0, 1)
_bench["completed_at"] = datetime.utcnow().isoformat()

results_dir = Path("..") / "bench" / "results"
results_dir.mkdir(exist_ok=True)
ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
bench_path = results_dir / f"04_lerobot_talkbot_{ts}.json"
bench_path.write_text(json.dumps(_bench, indent=2))
print(json.dumps(_bench, indent=2))
print(f"\nBenchmark saved: {bench_path}")